### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [493]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [494]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

### Data generator

##### Support functions

In [495]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [496]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.6, 1.5), (0.6, 0.65, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [497]:
############################### Function to generate the credit to be requested ##################################
# Note the credits will be exactly for one year
def generate_credit_requested(income): ### It will depend on the income
    # The will maximum enter as for a credit that represent between 25% and 150% of their income and the probabilities of any values are equal, so a uniform distribution
    percentage_of_monthly_income = random.uniform(0.25, 1.5)
    yearly_income = income * 12 # must be changes once dynamically made
    credit_requested_yearly = yearly_income *percentage_of_monthly_income
    credit_requested_monthly = credit_requested_yearly / 12
    percentage_credit_month_income = credit_requested_monthly/income
    return credit_requested_monthly, percentage_credit_month_income
    

In [498]:
####################### A function to calculate the collateral ########################
def calculate_collateral(profession):
    if "HighSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        # random.random() returns a float number between 0 and 1
        if random.random() < 0.10: # 10 % of the people do not have collateral
            return 0
        else: # 90 % have a collateral between 10,000 and 80,000
            return random.randint(10000, 80000)
    elif "MediumSkilled" in profession: # If the person is HighSkilled, even if currently unemployed
        if random.random() < 0.20: # 20 % of the people do not have collateral
            return 0
        else: # 80 % have a collateral between 10,000 and 50,000
            return random.randint(10000, 50000)
    else: # If Lowskilled
        if random.random() < 0.35: # 35 % of the people do not have collateral
            return 0
        else: # 65 % have a collateral between 5,000 and 25,000
            return random.randint(5000, 25000)

In [499]:
####################### A function to estimate the seizable assets to calculate the LGD ########################
def estimate_seizable_assets(monthly_income, savings, profession, collateral):
    
    # Base asset estimation as a portion of income and savings
    # This is a proxy, people with higher income, tend to have higher assets
    # If the savinds are negative, and higher than 2 times the monthy income this will draw this to be negative
    # In the end, if no assets can be seizured, it means that maybe it is not convenient for the bank to actually give the loan
    base_asset = savings + 2 * monthly_income

    # People that are High-Skilled tend to have more assets to be seized, even if they are currently unemployed
    if "HighSkilled" in profession:
        base_asset *= 1.2
    elif "MediumSkilled" in profession:
        base_asset *= 1.0
    else:
        base_asset *= 0.8

    # If they have a collateral, more sizes are
    base_asset += collateral

    # Add some noise to simulate unpredictability
    # That means given some processes in normal life, it is not always sure that 100% of the collateral can be recovered without cost
    base_asset *= np.random.normal(1, 0.1)  # 10% variation

    # To ensure that if nothing can be seized, because in the end a negative value is returned, then we get a zero
    return max(base_asset, 0)

In [500]:
############################### Function to generate the whether the person defaults or not ##################################
def generate_default_label(profession, past_credits, debt_to_income_ratio_before_credit, credit_to_income_ratio):
    """This simulates a default label (0/1) based on financial risk factors."""
    """What we will use will be the """
    
    # Configurable risk settings per profession
    risk_settings = {
        "Unemployed_LowSkilled":     {"base": 0.15, "weights": (0.30, 0.6, 0.4)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "LowSkilled":                {"base": 0.08, "weights": (0.20, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_MediumSkilled": {"base": 0.12, "weights": (0.30, 0.5, 0.3)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "MediumSkilled":            {"base": 0.05, "weights": (0.20, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "Unemployed_HighSkilled":   {"base": 0.09, "weights": (0.30, 0.45, 0.25)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
        "HighSkilled":              {"base": 0.2,  "weights": (0.20, 0.4, 0.2)}, # base probability of default and weights (past credits, debt to income before credit, credit to income)
    }

    # setting variable is sett to retrieve the element of reisk_settings for the profession given
    settings = risk_settings.get(profession)
    # In case there is the profession given for the function does not match any of the professions above listed
    if not settings:
        raise ValueError(f"Unknown profession: {profession}")
    
    # Defining the weigths for the calcualtion of the probability of default
    w1, w2, w3 = settings["weights"]
    # Defining the base probability of default
    base = settings["base"]

    # Calculating the risk factor with the weigths
    risk_factor = past_credits * w1 + debt_to_income_ratio_before_credit * w2 + credit_to_income_ratio * w3
    # Defining the default probability
    default_probability = min(1, base + risk_factor)

    # We want a non-deterministic y-categorical variable that will make that same profiles will not always lead to the same result
    # So even if two people may fall on the same profile, maybe they will not default
    # return 1 if random.random() < default_probability else 0
    return int(random.random() < default_probability)

##### Data Generator for the original state of individuals

In [501]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        
        # Debt to income ratio: PARTIAL, before the credit
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio_partial = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio_partial = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        monthly_credit = generate_credit_requested(monthly_income)[0]
        credit_to_income_ratio = generate_credit_requested(monthly_income)[1]
        
        # calculating the requested duration of the loan (maybe we can make it to be then also set by the bank whether it accepts it up to this term or not)
        if 0.25 <= credit_to_income_ratio < 0.5: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.2, 0.3, 0.2, 0.2, 0.01], k=1)[0]
        elif 0.5 <= credit_to_income_ratio < 0.75: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.1, 0.2, 0.2, 0.2, 0.3], k=1)[0]
        elif 0.75 <= credit_to_income_ratio < 1.0: 
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.08, 0.12, 0.2, 0.25, 0.35], k=1)[0]
        else:
            credit_term_months = random.choices([12, 24, 36, 48, 60], weights=[0.02, 0.08, 0.2, 0.2, 0.5], k=1)[0]
        
        # calculating the monthly debt if the monthluy credit is issued
        debt_after_credit = savings_debt - monthly_credit
        if debt_after_credit > 0: # if still the monthly savings are higher than the credit, then the total debt to incom ratio should be zero
            debt_to_income_ratio_total = 0
        else: 
            debt_to_income_ratio_total = abs(debt_after_credit/monthly_income)
            
        # collateral and seizurable asset
        collateral = calculate_collateral(profession)
        estimated_seizable_assets = estimate_seizable_assets(monthly_income, savings_debt, profession, collateral)
        
        # ---------------- Y-Variable --------------------------------#
        default_not_default = generate_default_label(profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio)
        
        data.append({
            'name': name, # independent
            'age': age, # independent
            'educational level': education_level, # independent
            'number of not paid past credits': past_credits, # independent
            'dependents': dependents, # independent
            'profession': profession, # depends on education
            'monthly income': monthly_income, # depends on profession and age
            'monthly expenditure': monthly_expenditure, # depends on income, mean income per age and profession, and the number of dependents
            'savings (debt)': savings_debt, # monthly income - monthly expenditure
            'debt-to-income ratio before credit': debt_to_income_ratio_partial, # abs(savings_debt/monthly_income)
            'credit: monthly amount': monthly_credit, # depends on the income
            'credit-to-income ratio': credit_to_income_ratio, # credit/income
            'requested_loan_duration': credit_term_months, # depends on the credit-to-income ratio
            'debt-to-income ratio after credit': debt_to_income_ratio_total, # abs((savings_debt - credit)/monthly_income)
            'collateral': collateral, # depends on the profession
            'estimated seizable assets': estimated_seizable_assets, # it is based on the monthly income, the savings, the profession and the collateral
            'y-categorical-default': default_not_default # depending on profession, past_credits, debt_to_income_ratio_partial, credit_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [502]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
0,name0,39,bachelor degree,1,0,HighSkilled,3482.0,3931.584087,-449.584087,0.129117,1493.878692,1.449448,60,0.558146,21801,32639.763191,1
1,name1,52,ausbildung,0,1,MediumSkilled,2271.0,3217.042830,-946.042830,0.416575,2899.495414,1.456416,60,1.693324,49078,51404.521076,0
2,name2,59,bachelor degree,0,1,HighSkilled,10034.0,6287.700407,3746.299593,0.000000,3564.399636,1.294604,24,0.000000,24724,49030.395500,0
3,name3,35,bachelor degree,1,1,HighSkilled,2461.0,2352.463166,108.536834,0.000000,1984.067183,0.548294,48,0.762101,24568,29936.943438,1
4,name4,49,high school or lower,0,2,LowSkilled,1793.0,1680.213909,112.786091,0.000000,1387.216262,0.583960,36,0.710781,0,3283.959425,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,46,bachelor degree,0,0,Unemployed_HighSkilled,3500.0,2261.581205,1238.418795,0.000000,4883.824222,1.039244,60,1.041544,0,9228.426709,0
996,name996,48,ausbildung,2,0,MediumSkilled,3824.0,2020.761327,1803.238673,0.000000,4696.401596,0.402678,36,0.756580,24715,38884.909140,1
997,name997,37,high school or lower,1,0,LowSkilled,1244.0,726.436919,517.563081,0.000000,1479.063271,0.649463,24,0.772910,0,2607.606194,0
998,name998,58,bachelor degree,1,0,Unemployed_HighSkilled,4500.0,3336.437773,1163.562227,0.000000,2587.993600,1.343046,60,0.316540,76571,82223.190103,1


##### DF statistics

In [503]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.771000,0.522000,0.573000,3646.405000,3356.674071,289.730929,0.105625,3133.009099,0.868373,42.672000,0.818949,27373.724000,35873.932215,0.474000
std,8.731942,0.689922,0.831482,2589.569733,2536.708064,1531.872692,0.228277,2727.546611,0.358286,15.922624,0.484128,22545.764821,26571.881498,0.499573
min,30.000000,0.000000,0.000000,500.000000,354.251387,-10540.628011,0.000000,154.783613,0.253610,12.000000,0.000000,0.000000,693.238754,0.000000
25%,37.000000,0.000000,0.000000,1741.750000,1546.954956,-241.403777,0.000000,1244.933219,0.566073,33.000000,0.433691,9500.000000,14680.387863,0.000000
50%,45.000000,0.000000,0.000000,2736.000000,2586.470112,227.042552,0.000000,2228.300185,0.869059,48.000000,0.804302,22751.000000,30232.915408,0.000000
75%,52.000000,1.000000,1.000000,5122.750000,4455.287318,795.235884,0.108752,4152.102914,1.183438,60.000000,1.157557,43059.000000,52390.318532,1.000000
max,60.000000,3.000000,4.000000,14039.000000,17421.628011,9170.047349,2.071802,20656.852280,1.496422,60.000000,2.542299,79877.000000,112257.042115,1.000000


Monthly income by profession

In [504]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,6245.212366,2381.481663,1500.0,4402.0,6040.0,7819.25,14039.0
LowSkilled,260.0,1588.169231,498.050433,543.0,1191.0,1573.0,1926.00,3554.0
MediumSkilled,257.0,2730.202335,876.607201,1152.0,2062.0,2560.0,3305.00,5577.0
Unemployed_HighSkilled,46.0,2956.521739,1071.479180,1500.0,1687.5,3000.0,3500.00,4500.0
Unemployed_LowSkilled,30.0,731.666667,209.055072,500.0,525.0,750.0,975.00,1100.0
Unemployed_MediumSkilled,35.0,1447.142857,402.737481,900.0,1150.0,1300.0,1875.00,2000.0


Debt-to-income ratio by profession

In [505]:
df_1.groupby('profession')['debt-to-income ratio before credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,0.112869,0.257013,0.0,0.0,0.0,0.111951,2.071802
LowSkilled,260.0,0.100384,0.205721,0.0,0.0,0.0,0.106098,1.606936
MediumSkilled,257.0,0.126490,0.244471,0.0,0.0,0.0,0.156598,1.168240
Unemployed_HighSkilled,46.0,0.054128,0.091476,0.0,0.0,0.0,0.112680,0.364327
Unemployed_LowSkilled,30.0,0.036223,0.075236,0.0,0.0,0.0,0.017042,0.241048
Unemployed_MediumSkilled,35.0,0.041528,0.076874,0.0,0.0,0.0,0.042340,0.311770


In [506]:
df_1.groupby('profession')['debt-to-income ratio after credit'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,0.821420,0.481963,0.000000,0.432164,0.811343,1.145439,2.498079
LowSkilled,260.0,0.817616,0.503068,0.000000,0.421111,0.813427,1.172375,2.075862
MediumSkilled,257.0,0.821622,0.499786,0.000000,0.435049,0.789474,1.174076,2.542299
Unemployed_HighSkilled,46.0,0.788672,0.405688,0.000000,0.444981,0.762143,1.041788,1.542099
Unemployed_LowSkilled,30.0,0.724585,0.372492,0.018070,0.537688,0.664737,0.938312,1.504114
Unemployed_MediumSkilled,35.0,0.903648,0.436555,0.060216,0.673527,0.862527,1.288035,1.761834


In [507]:
df_1.groupby('profession')['credit-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,0.887365,0.362619,0.255812,0.587442,0.866343,1.213551,1.488987
LowSkilled,260.0,0.865650,0.348387,0.253844,0.574776,0.878777,1.160538,1.487829
MediumSkilled,257.0,0.848178,0.361376,0.253610,0.542048,0.838271,1.136121,1.496422
Unemployed_HighSkilled,46.0,0.877019,0.355767,0.278637,0.608323,0.895996,1.178051,1.443771
Unemployed_LowSkilled,30.0,0.813298,0.376675,0.259026,0.465538,0.816089,1.070906,1.491450
Unemployed_MediumSkilled,35.0,0.870890,0.361498,0.305583,0.554928,0.938888,1.196829,1.458128


In [508]:
df_1.groupby('profession')['collateral'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,40247.607527,23738.197482,0.0,21041.0,40536.5,61553.75,79637.0
LowSkilled,260.0,9462.323077,8622.708853,0.0,0.0,8518.0,17237.50,24788.0
MediumSkilled,257.0,24800.272374,16404.904750,0.0,12800.0,26271.0,39520.00,49987.0
Unemployed_HighSkilled,46.0,49400.130435,23496.660547,0.0,32305.0,55871.0,66761.25,79877.0
Unemployed_LowSkilled,30.0,12390.666667,8658.212706,0.0,5586.5,14611.0,19585.50,24710.0
Unemployed_MediumSkilled,35.0,26388.971429,15067.326748,0.0,17776.5,28229.0,39252.00,49927.0


In [509]:
df_1.groupby('profession')['estimated seizable assets'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,56026.724104,25192.200888,3131.487017,36090.752333,54329.640420,75510.718756,112257.042115
LowSkilled,260.0,12068.569286,8759.698312,789.345908,3315.730779,10972.579533,19427.113622,33412.186343
MediumSkilled,257.0,30632.454542,17118.812460,1747.083071,17302.313024,31986.412447,44380.745654,72339.392557
Unemployed_HighSkilled,46.0,56239.104452,22926.731846,9228.426709,39886.275977,60818.902625,76013.010877,87867.358829
Unemployed_LowSkilled,30.0,13859.260173,8902.398170,693.238754,6991.783424,15768.835627,20193.169856,27392.791741
Unemployed_MediumSkilled,35.0,29109.868772,15676.043717,2340.923050,20398.912729,28924.132189,42093.079486,66311.721195


In [510]:
df_1.groupby('profession')['y-categorical-default'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,372.0,0.478495,0.500210,0.0,0.0,0.0,1.0,1.0
LowSkilled,260.0,0.503846,0.500949,0.0,0.0,1.0,1.0,1.0
MediumSkilled,257.0,0.424125,0.495174,0.0,0.0,0.0,1.0,1.0
Unemployed_HighSkilled,46.0,0.478261,0.505047,0.0,0.0,0.0,1.0,1.0
Unemployed_LowSkilled,30.0,0.600000,0.498273,0.0,0.0,1.0,1.0,1.0
Unemployed_MediumSkilled,35.0,0.457143,0.505433,0.0,0.0,0.0,1.0,1.0


# MODELLING PD, LGD, EAD

## PD (probability of default)
The probability that a customer with default at some point

#### Packages

In [511]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Converting variables into dummies

In [512]:
# Convert categorical variables to numeric
df_R = pd.get_dummies(df_1, columns=["educational level", "profession"], drop_first=True)
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,estimated seizable assets,y-categorical-default,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled
0,name0,39,1,0,3482.0,3931.584087,-449.584087,0.129117,1493.878692,1.449448,...,32639.763191,1,True,False,False,False,False,False,False,False
1,name1,52,0,1,2271.0,3217.042830,-946.042830,0.416575,2899.495414,1.456416,...,51404.521076,0,False,False,False,False,True,False,False,False
2,name2,59,0,1,10034.0,6287.700407,3746.299593,0.000000,3564.399636,1.294604,...,49030.395500,0,True,False,False,False,False,False,False,False
3,name3,35,1,1,2461.0,2352.463166,108.536834,0.000000,1984.067183,0.548294,...,29936.943438,1,True,False,False,False,False,False,False,False
4,name4,49,0,2,1793.0,1680.213909,112.786091,0.000000,1387.216262,0.583960,...,3283.959425,0,False,True,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,46,0,0,3500.0,2261.581205,1238.418795,0.000000,4883.824222,1.039244,...,9228.426709,0,True,False,False,False,False,True,False,False
996,name996,48,2,0,3824.0,2020.761327,1803.238673,0.000000,4696.401596,0.402678,...,38884.909140,1,False,False,False,False,True,False,False,False
997,name997,37,1,0,1244.0,726.436919,517.563081,0.000000,1479.063271,0.649463,...,2607.606194,0,False,True,False,True,False,False,False,False
998,name998,58,1,0,4500.0,3336.437773,1163.562227,0.000000,2587.993600,1.343046,...,82223.190103,1,True,False,False,False,False,True,False,False


#### Defining X and y

In [513]:
# Column "name" is dropped from the dataframe, no need to keep it
# All the variables except y-categorical-default are X
X = df_R.drop(columns=["name", "y-categorical-default"])
y = df_R["y-categorical-default"]

#### Split between trainning and test sets

In [514]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#### Standardize the variables for a better gradient descendt
Standardizing features to have a mean of zero ensures that all features are centered around the same baseline, which helps prevent models from being biased toward features with larger numerical values. It also makes gradient-based optimization methods like gradient descent behave more efficiently by ensuring all features contribute equally to the cost function.

In [515]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### LOGIT

##### Packages

In [516]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##### Training the Logistic Regression

In [517]:
log_reg = LogisticRegression()
log_reg.fit(X_train_scaled, y_train)

LogisticRegression()

##### Predictions

In [518]:
y_pred = log_reg.predict(X_test_scaled)

##### Evaluations of the accuracy of model

In [519]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.685


In [520]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.65      0.85      0.74       105
           1       0.75      0.51      0.60        95

    accuracy                           0.69       200
   macro avg       0.70      0.68      0.67       200
weighted avg       0.70      0.69      0.67       200



##### Estimating the PDs

In [521]:
# df_R["PD_LR"] = log_reg.predict_proba(scaler.transform(X))[:, 1]
# df_R


### Using Neuronal Networks

##### Packages

In [522]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

##### Building the Neuronal Network

In [523]:
# We may try out:

# tanh: The hyperbolic tangent function outputs values between -1 and 1, making it useful for hidden layers 
# where you want activations that are zero-centered, which helps in faster convergence and avoids saturation for small inputs.
 
# relu: The Rectified Linear Unit activation function outputs zero for any negative input and passes positive values as they are.
# It is widely used in hidden layers for its simplicity and effectiveness, and helps avoid the vanishing gradient problem seen with functions like sigmoid and tanh.

# softmax: Softmax is typically used in the output layer for multi-class classification tasks. It converts the raw outputs into probabilities, 
# ensuring that the sum of all output values equals 1, representing the probability distribution over multiple classes.

# This is like having an input which is your variable X, then 32 neurons process the input features,
# using a function (in this case "tanh") to calculate the weights and transformations at each neuron. 
# The results are then passed to a subsequent layer with 16 neurons, where again "tanh" is applied to further transform the data.
# In the end, everything is passed through a final neuron that uses a "sigmoid" (logistic) function 
# to produce an output between [0, 1], representing a probability for binary classification.
model = Sequential([  # Each layer is run after the other, forming a linear stack of layers.
    # The first Dense layer applies 32 units (neurons) and uses the "tanh" activation function.
    # The input_shape corresponds to the number of features in the dataset (X_train_scaled).
    Dense(32, activation='tanh', input_shape=(X_train_scaled.shape[1],)),
    
    # The second Dense layer applies 16 units (neurons) and uses "tanh" activation function.
    # "tanh" ensures that the output of each neuron will be between -1 and 1, centering the activations.
    Dense(16, activation='tanh'),
    
    # The final Dense layer outputs a single value, which is the probability of the positive class.
    # Sigmoid activation squashes the output to a value between 0 and 1.
    # This is commonly used for binary classification, where the output is a probability of class 1.
    Dense(1, activation='sigmoid')
])

/Users/bonjour/opt/anaconda3/envs/bankgame/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##### Compiling the model

In [524]:
# The model is being compiled with the following parameters:
# optimizer='adam': The Adam optimizer is being used. It is an adaptive learning rate optimization algorithm that 
#  combines the benefits of both AdaGrad and RMSProp, making it well-suited for most deep learning models.
# loss='binary_crossentropy': The loss function used is binary cross-entropy, which is appropriate for binary classification 
#  tasks where the output is a probability of belonging to one of two classes, which is the case of our y-variable
# metrics=['accuracy']: The model will track accuracy as the evaluation metric during training and testing, 
#  which measures the percentage of correct predictions.

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

##### Training the model

In [525]:
# The model is fit with X_train_scaled: This means the model is being trained on the scaled training data (X_train_scaled) 
# using the corresponding labels (y_train).

# Uses 25 epochs: An epoch refers to one full pass through the entire training dataset. 
# The model will train for 25 epochs, meaning it will go through the data 25 times to learn the optimal weights.

# It will use a batch size of 32: The model will train using 32 samples (or rows of data) at a time, and after processing 
# those 32, it updates the weights before moving on to the next 32 samples. 

# During training, the model's performance is periodically evaluated on the validation set (X_test_scaled and y_test) 
# to monitor overfitting and to adjust the training accordingly.

model_NN = model.fit(X_train_scaled, y_train, epochs=50, batch_size=32, validation_data=(X_test_scaled, y_test))

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5069 - loss: 0.7193 - val_accuracy: 0.6350 - val_loss: 0.6462
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6180 - loss: 0.6499 - val_accuracy: 0.6750 - val_loss: 0.6138
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6655 - loss: 0.6235 - val_accuracy: 0.6750 - val_loss: 0.6053
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6771 - loss: 0.6091 - val_accuracy: 0.6900 - val_loss: 0.5992
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6754 - loss: 0.6104 - val_accuracy: 0.6850 - val_loss: 0.5968
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6983 - loss: 0.5980 - val_accuracy: 0.6800 - val_loss: 0.5957
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6742 - loss: 0.6075 - val_accuracy: 0.6750 - val_loss: 0.5951
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7199 - loss: 0.5767 - val_accuracy: 0.6900 - val_loss

##### We want to get the last accuracy of the last epoch and the minimal and highest accuracy values for comparison

In [526]:
train_accuracies_NN = model_NN.history['accuracy']

In [527]:
# final accuracy value
final_accuracy = train_accuracies_NN[-1]
# maximal accuracy value
min_accuracy = min(train_accuracies_NN)
# minimal accuracy value
max_accuracy = max(train_accuracies_NN)
# average accuracy value
average_accuracy = sum(train_accuracies_NN) / len(train_accuracies_NN)

In [528]:
# Print the results
print(f'Final accuracy: {final_accuracy:.4f}')
print(f'Minimum accuracy during training of the NN: {min_accuracy:.4f}')
print(f'Maximum accuracy during training of the NN: {max_accuracy:.4f}')
print(f'Average accuracy during training of the NN: {average_accuracy:.4f}')

Final accuracy: 0.7700
Minimum accuracy during training of the NN: 0.5462
Maximum accuracy during training of the NN: 0.7700
Average accuracy during training of the NN: 0.7163


##### Estimating the PDs

In [529]:
#df_R["PD_NN"]= model.predict(scaler.transform(X))
#df_R

### Using Random Forests

##### Packages

In [530]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

##### Fitting the model

In [531]:
# Initialize the RandomForestClassifier with the following parameters:
# n_estimators=100: This sets the number of decision trees (estimators) in the forest. The model will train 100 individual trees and aggregate their results to make predictions.
# random_state=42: This ensures reproducibility by fixing the random seed used in the training process. 
# To get the same result even if the code is run multiple times (obviously this will only be affected by the random nature of our data)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

RandomForestClassifier(random_state=42)

##### Accuracy of the model

In [532]:
# Predict class labels for X_test_scaled
y_pred = rf_model.predict(X_test_scaled)
# Calculate accuracy by comparing the predicted labels with our simulated data y
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.6800


##### Estimating the PDs

In [533]:
# df_R = df_R.iloc[:len(X_test_scaled)]
# df_R["PD_RF"] = rf_model.predict_proba(X_test_scaled)[:, 1]
#df_R["PD_RF"] = rf_model.predict_proba(scaler.transform(X))[:, 1] # needs to be checked
#df_R

### Using Logistic LASSO

#### Packages

In [534]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

#### Parameter of the regularization strenght

In [535]:
alpha = 0.01  # Regularization strength

#### Training the regression

In [536]:
# L1-regularized logistic regression (like lasso but for classification problems)
logistic_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1/alpha, random_state=42)

# Fit the model
logistic_lasso.fit(X_train_scaled, y_train)


LogisticRegression(C=100.0, penalty='l1', random_state=42, solver='liblinear')

In [537]:
logistic_lasso.coef_

array([[ 0.00220619,  0.85366755,  0.0875721 , -0.06299908, -0.14907593,
        -0.04138499,  0.52256159, -0.17260281,  0.47368884, -0.08217477,
         0.05464209, -0.40804061,  0.37059324,  0.18212093, -0.02949513,
         0.08654379, -0.01630456, -0.10668709, -0.06668086,  0.1007504 ,
        -0.01366912]])

#### Making predictions with the regression

In [538]:
# Predict probabilities
y_pred_proba = logistic_lasso.predict_proba(X_test_scaled)[:, 1]  # Probabilities of class 1

# Apply threshold at 0.5 to get predicted class labels
y_pred_class = (y_pred_proba >= 0.5).astype(int)

Getting accuracy measures

In [539]:
# Accuracy by rule of PD >= 0.5 default and PD < 0.5 not default
y_pred_class = (y_pred_proba >= 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred_class)
print(f"Accuracy: {accuracy:.4f}")

# AUC score (for probability-based performance)
auc = roc_auc_score(y_test, y_pred_proba)
print(f"AUC: {auc:.4f}")

Accuracy: 0.6900
AUC: 0.7595


#### Now estimating the PDs for our whole data set

In [540]:
#df_R["PD_LL"] = logistic_lasso.predict_proba(scaler.transform(X))[:, 1]

### Now getting the general statistics of what I did, for all the four models: Logisitc regression, Neuronal networks, Random Forests and Logistic LASSO

In [541]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.771000,0.522000,0.573000,3646.405000,3356.674071,289.730929,0.105625,3133.009099,0.868373,42.672000,0.818949,27373.724000,35873.932215,0.474000
std,8.731942,0.689922,0.831482,2589.569733,2536.708064,1531.872692,0.228277,2727.546611,0.358286,15.922624,0.484128,22545.764821,26571.881498,0.499573
min,30.000000,0.000000,0.000000,500.000000,354.251387,-10540.628011,0.000000,154.783613,0.253610,12.000000,0.000000,0.000000,693.238754,0.000000
25%,37.000000,0.000000,0.000000,1741.750000,1546.954956,-241.403777,0.000000,1244.933219,0.566073,33.000000,0.433691,9500.000000,14680.387863,0.000000
50%,45.000000,0.000000,0.000000,2736.000000,2586.470112,227.042552,0.000000,2228.300185,0.869059,48.000000,0.804302,22751.000000,30232.915408,0.000000
75%,52.000000,1.000000,1.000000,5122.750000,4455.287318,795.235884,0.108752,4152.102914,1.183438,60.000000,1.157557,43059.000000,52390.318532,1.000000
max,60.000000,3.000000,4.000000,14039.000000,17421.628011,9170.047349,2.071802,20656.852280,1.496422,60.000000,2.542299,79877.000000,112257.042115,1.000000


# We need here a Montecarlo to get the model with the highest accurracy results and they will be then added to the df, the others wont be added, but will remain part of the code.

In [542]:
from sklearn.metrics import roc_auc_score, brier_score_loss
import numpy as np

In [543]:
results = {'LR': [], 'NN': [], 'RF': [], 'LL': []}

In [544]:
for _ in range(500): # Monte Carlo simulation 500 times
    
    df = data_generator(1000)
    
    # DATA TRANSFORMATION AND DEFINITION -----------------------------------------------------------
    # Transforming the data to be for the modelling
    df = pd.get_dummies(df, columns=["educational level", "profession"], drop_first=True)
    
    # Setting the X and y variables
    X = df.drop(columns=["name", "y-categorical-default"])
    y = df["y-categorical-default"]
    
    # We take the above defined models to make predictions
    
    # GETTING THE MonteCarlo MODELS PREDICTIONS --------------------------------------------------------------------------
    
    # Logisitic regression
    y_pred_lr = log_reg.predict_proba(scaler.transform(X))[:, 1]
    results['LR'].append(brier_score_loss(y, y_pred_lr))
    
    # Neural network
    y_pred_nn = model.predict(scaler.transform(X))
    results['NN'].append(brier_score_loss(y, y_pred_nn))
    
    # Random forest
    y_pred_rf = rf_model.predict_proba(scaler.transform(X))[:, 1]
    results['RF'].append(brier_score_loss(y, y_pred_rf))
    
    # Logistic regression with L1 regularization
    y_pred_ll = logistic_lasso.predict_proba(scaler.transform(X))[:, 1]
    results['LL'].append(brier_score_loss(y, y_pred_ll))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


In [545]:
# Dictionary with the mean accuracy values
PD_models_average_accuracy =  {'LR': [], 'NN': [], 'RF': [], 'LL': []}
# Apppend the values
PD_models_average_accuracy['LR'].append(np.mean(results['LR']))
PD_models_average_accuracy['NN'].append(np.mean(results['NN']))
PD_models_average_accuracy['RF'].append(np.mean(results['RF']))
PD_models_average_accuracy['LL'].append(np.mean(results['LL']))
print("Average Brier score loss over 500 iterations:")
print(f"Logistic Regression: {PD_models_average_accuracy['LR'][-1]:.4f}")
print(f"Neural Network: {PD_models_average_accuracy['NN'][-1]:.4f}")
print(f"Random Forest: {PD_models_average_accuracy['RF'][-1]:.4f}")
print(f"Logistic Regression with L1 Regularization: {PD_models_average_accuracy['LL'][-1]:.4f}")
# Brier score measures the mean squared difference between predicted probabilities and actual outcomes.
# Lower Brier scores indicate better calibration and accuracy of predicted probabilities.
# Brier score values range from 0 to 1, with lower values indicating better performance.
# It is sensitive to the absolute values of probabilities, making it useful for assessing the quality of probability estimates.
# Brier score is particularly useful for evaluating probabilistic predictions in binary classification tasks.           

Average Brier score loss over 500 iterations:
Logistic Regression: 0.2136
Neural Network: 0.2304
Random Forest: 0.2259
Logistic Regression with L1 Regularization: 0.2139


#### Let's find out the best model

In [546]:
# Find the model with the highest average accuracy
best_model = min(PD_models_average_accuracy, key=lambda k: PD_models_average_accuracy[k][-1])
print(f"The best model is: {best_model} with an average AUC score of {PD_models_average_accuracy[best_model][-1]:.4f}")

The best model is: LR with an average AUC score of 0.2136


#### Now let's set the final value of the PD in my originally generated Database given the results of the MonteCarlo accuracy

In [547]:
if best_model == 'LR':
    df_R["PD"] = log_reg.predict_proba(scaler.transform(X))[:, 1]
elif best_model == 'NN':
    df_R["PD"]= model.predict(scaler.transform(X))
elif best_model == 'RF':    
    df_R["PD"] = rf_model.predict_proba(scaler.transform(X))[:, 1]
elif best_model == 'LL':
    df_R["PD"] = logistic_lasso.predict_proba(scaler.transform(X))[:, 1]
else:
    raise ValueError("Unknown model type")       
    

## EAD (exposure at default)
It is the total amount at risk at the moment the borrower defaults

That is the amount fo the credit that has been unpaid at the moment of the calculation

In [548]:
df_R["EAD"] = df_R["credit: monthly amount"] * 12
df_R

,name,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,...,educational level_bachelor degree,educational level_high school or lower,educational level_post graduate degree,profession_LowSkilled,profession_MediumSkilled,profession_Unemployed_HighSkilled,profession_Unemployed_LowSkilled,profession_Unemployed_MediumSkilled,PD,EAD
0,name0,39,1,0,3482.0,3931.584087,-449.584087,0.129117,1493.878692,1.449448,...,True,False,False,False,False,False,False,False,0.905474,17926.544302
1,name1,52,0,1,2271.0,3217.042830,-946.042830,0.416575,2899.495414,1.456416,...,False,False,False,False,True,False,False,False,0.280794,34793.944974
2,name2,59,0,1,10034.0,6287.700407,3746.299593,0.000000,3564.399636,1.294604,...,True,False,False,False,False,False,False,False,0.651061,42772.795637
3,name3,35,1,1,2461.0,2352.463166,108.536834,0.000000,1984.067183,0.548294,...,True,False,False,False,False,False,False,False,0.202837,23808.806197
4,name4,49,0,2,1793.0,1680.213909,112.786091,0.000000,1387.216262,0.583960,...,False,True,False,True,False,False,False,False,0.201211,16646.595144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,name995,46,0,0,3500.0,2261.581205,1238.418795,0.000000,4883.824222,1.039244,...,True,False,False,False,False,True,False,False,0.652981,58605.890669
996,name996,48,2,0,3824.0,2020.761327,1803.238673,0.000000,4696.401596,0.402678,...,False,False,False,False,True,False,False,False,0.220842,56356.819153
997,name997,37,1,0,1244.0,726.436919,517.563081,0.000000,1479.063271,0.649463,...,False,True,False,True,False,False,False,False,0.197696,17748.759247
998,name998,58,1,0,4500.0,3336.437773,1163.562227,0.000000,2587.993600,1.343046,...,True,False,False,False,False,True,False,False,0.347791,31055.923203


## LGD (loss given default)

Example:

1. A borrower takes a loan of €10,000. 
2. They default after paying back €2,000, and the bank recovers €4,000 by seizing assets. 

That means:

Total Recovered = €2,000 (paid) + €4,000 (recovered from assets) = €6,000

Total Loss = €10,000 - €6,000 = €4,000

LGD = €4,000 (Total Loss)/ €10,000 (Loan Amount) = 40%



#### In our approximation 
LGD = (EAD - what can be sized)/EAD = 1 - (What can be seized/EAD)

In [549]:
def estimate_lgd(EAD, seizable_assets):
    lgd = 1 - (seizable_assets / EAD)
    return max(0, min(lgd, 1))  # The lgd should be between 0 and 1

In [550]:
df_R["LGD"] = df_R.apply(lambda row: estimate_lgd(row["EAD"], row["estimated seizable assets"]), axis=1) # This should be applied row by row

## EL (expected loss)

EL=PD×LGD×EAD

In [551]:
#df_R["EL"] = df_R["LGD"] * df_R["EAD"] * df_R["PD_NN"] # This is the expected loss, given the PD of the NN model
df_R["EL"] = df_R["LGD"] * df_R["EAD"] * df_R["PD"] # This is the expected loss

## We want here to estimate a suggested interest rate that should be charged to the customer

#### Inputs of the function

In [552]:
base_rate = 0.03  # central bank or risk-free rate
max_rate = 0.45 # maximum legal rate to be charged
#PD = df_R["PD_NN"]  # Probability of default
PD = df_R["PD"] # Probability of default
LGD = df_R["LGD"]  # Loss given default
bank_margin = 0.02  # Bank's margin

#### Function

In [553]:
def calculate_suggested_rate(base_rate, max_rate, PD, LGD, margin):
    # Calculate the risk premium
    risk_premium = PD * LGD
    # Calculate the suggested rate
    suggested_rate = base_rate + risk_premium + margin
    return min(suggested_rate, max_rate)

#### Estimation of the calculated rate

In [554]:
df_R["Suggested interest rate"] = df_R.apply(
    lambda row: calculate_suggested_rate(base_rate, max_rate, row["PD"], row["LGD"], bank_margin), axis=1
)

## Making one last description of the columns

In [555]:
df_R.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio before credit,credit: monthly amount,credit-to-income ratio,requested_loan_duration,debt-to-income ratio after credit,collateral,estimated seizable assets,y-categorical-default,PD,EAD,LGD,EL,Suggested interest rate
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,44.771000,0.522000,0.573000,3646.405000,3356.674071,289.730929,0.105625,3133.009099,0.868373,42.672000,0.818949,27373.724000,35873.932215,0.474000,0.473924,37596.109184,0.256664,5687.485383,0.155285
std,8.731942,0.689922,0.831482,2589.569733,2536.708064,1531.872692,0.228277,2727.546611,0.358286,15.922624,0.484128,22545.764821,26571.881498,0.499573,0.232166,32730.559336,0.323287,11614.319048,0.141259
min,30.000000,0.000000,0.000000,500.000000,354.251387,-10540.628011,0.000000,154.783613,0.253610,12.000000,0.000000,0.000000,693.238754,0.000000,0.069443,1857.403353,0.000000,0.000000,0.050000
25%,37.000000,0.000000,0.000000,1741.750000,1546.954956,-241.403777,0.000000,1244.933219,0.566073,33.000000,0.433691,9500.000000,14680.387863,0.000000,0.277761,14939.198632,0.000000,0.000000,0.050000
50%,45.000000,0.000000,0.000000,2736.000000,2586.470112,227.042552,0.000000,2228.300185,0.869059,48.000000,0.804302,22751.000000,30232.915408,0.000000,0.436031,26739.602216,0.000000,0.000000,0.050000
75%,52.000000,1.000000,1.000000,5122.750000,4455.287318,795.235884,0.108752,4152.102914,1.183438,60.000000,1.157557,43059.000000,52390.318532,1.000000,0.655134,49825.234970,0.519693,6491.833749,0.245264
max,60.000000,3.000000,4.000000,14039.000000,17421.628011,9170.047349,2.071802,20656.852280,1.496422,60.000000,2.542299,79877.000000,112257.042115,1.000000,0.997944,247882.227358,0.934280,110833.573635,0.450000
